In [1]:
# 1. Tải Source code PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd /content/PaddleOCR

# 2. Cài đặt framework PaddlePaddle hỗ trợ GPU
!python -m pip install paddlepaddle-gpu -i https://mirror.baidu.com/pypi/simple

# 3. Cài đặt các thư viện phụ trợ
!pip install -r requirements.txt
!pip install rapidfuzz pyclipper lmdb

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 338157, done.
remote: Counting objects: 100% (1086/1086), done.
remote: Compressing objects: 100% (296/296), done.
remote: Total 338157 (delta 918), reused 790 (delta 790), pack-reused 337071 (from 3)
Receiving objects: 100% (338157/338157), 1.79 GiB | 18.10 MiB/s, done.
Resolving deltas: 100% (267860/267860), done.
/content/PaddleOCR
Looking in indexes: https://mirror.baidu.com/pypi/simple
ERROR: Could not find a version that satisfies the requirement paddlepaddle-gpu (from versions: none)
ERROR: No matching distribution found for paddlepaddle-gpu
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.0 MB/s eta 0:00:00


In [3]:
!pip install paddlepaddle-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [3]:
import os
import cv2
import numpy as np
import json
from google.colab import drive

# Mount Google Drive để lưu model sau khi train
drive.mount('/content/drive')

# Tạo thư mục chứa data cho phần Recognition
rec_data_dir = '/content/PaddleOCR/train_data_rec/images'
os.makedirs(rec_data_dir, exist_ok=True)

# 1. Tạo vài bức ảnh đen nhỏ (mô phỏng ảnh đã được crop vừa khít chữ)
img1 = np.zeros((32, 100, 3), dtype=np.uint8)
img2 = np.zeros((32, 120, 3), dtype=np.uint8)

cv2.imwrite(os.path.join(rec_data_dir, 'crop_001.jpg'), img1)
cv2.imwrite(os.path.join(rec_data_dir, 'crop_002.jpg'), img2)

# 2. Tạo file rec_label.txt (Định dạng: đường_dẫn_ảnh \t Text)
label_path = '/content/PaddleOCR/train_data_rec/rec_label.txt'
with open(label_path, 'w', encoding='utf-8') as f:
    f.write("images/crop_001.jpg\t日本語\n")
    f.write("images/crop_002.jpg\t辞書アプリ\n")

print("✅ Đã tạo xong Mini-Dataset cho phần Recognition!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã tạo xong Mini-Dataset cho phần Recognition!


In [5]:
%cd /content/PaddleOCR
!mkdir -p pretrain_models

# Tải pre-trained model tiếng Nhật từ server của Paddle
!wget -P ./pretrain_models/ https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/japan_PP-OCRv3_rec_train.tar

# Giải nén
!tar -xf ./pretrain_models/japan_PP-OCRv3_rec_train.tar -C ./pretrain_models/
print("✅ Đã tải và giải nén Pre-trained Model!")

/content/PaddleOCR
--2026-05-05 09:28:47--  https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/japan_PP-OCRv3_rec_train.tar
Resolving paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddleocr.bj.bcebos.com (paddleocr.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 314255360 (300M) [application/x-tar]
Saving to: ‘./pretrain_models/japan_PP-OCRv3_rec_train.tar’

japan_PP-OCRv3_rec_ 100%[===================>] 299.70M  24.6MB/s    in 13s     

2026-05-05 09:29:01 (22.6 MB/s) - ‘./pretrain_models/japan_PP-OCRv3_rec_train.tar’ saved [314255360/314255360]

✅ Đã tải và giải nén Pre-trained Model!


In [22]:
import os
import shutil

# 1. Cài đặt Kaggle API (Đảm bảo bạn đã upload file kaggle.json lên thư mục gốc /content/)
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Tải và giải nén bộ ICDAR 2019 MLT (sẽ mất vài phút)
print("Đang tải dữ liệu 14GB từ Kaggle...")
!kaggle datasets download -d zubairalibhutto/mlt-19-ocr-dataset
print("Đang giải nén dữ liệu...")
!unzip -q mlt-19-ocr-dataset.zip -d /content/ICDAR2019
print("✅ Tải và giải nén hoàn tất!")

Đang tải dữ liệu 14GB từ Kaggle...
Dataset URL: https://www.kaggle.com/datasets/zubairalibhutto/mlt-19-ocr-dataset
License(s): MIT
100% 13.3G/13.3G [11:26<00:00, 20.9MB/s]

Đang giải nén dữ liệu...
✅ Tải và giải nén hoàn tất!


In [4]:
import os
import cv2
import numpy as np

# Đường dẫn bộ dữ liệu gốc
raw_img_dir = '/content/ICDAR2019/TrainImages/TrainImages'
raw_gt_dir = '/content/ICDAR2019/TrainGT/TrainGT'

# Thư mục đích cho mô hình Recognition
rec_img_dir = '/content/PaddleOCR/train_data_rec/images'
rec_label_file = '/content/PaddleOCR/train_data_rec/rec_label.txt'

os.makedirs(rec_img_dir, exist_ok=True)

print("Bắt đầu xử lý và cắt (crop) ảnh tiếng Nhật...")
crop_count = 0

with open(rec_label_file, 'w', encoding='utf-8') as f_out:
    for gt_filename in os.listdir(raw_gt_dir):
        if not gt_filename.endswith('.txt'): continue

        gt_filepath = os.path.join(raw_gt_dir, gt_filename)
        img_filename = gt_filename.replace('.txt', '.jpg')
        if img_filename.startswith('gt_') and not os.path.exists(os.path.join(raw_img_dir, img_filename)):
            img_filename = img_filename.replace('gt_', '')

        img_filepath = os.path.join(raw_img_dir, img_filename)
        if not os.path.exists(img_filepath): continue

        img = cv2.imread(img_filepath)
        if img is None: continue

        # Đọc file nhãn gốc
        with open(gt_filepath, 'r', encoding='utf-8-sig') as f_in:
            for line in f_in:
                parts = line.strip().split(',')
                if len(parts) >= 10:
                    language = parts[8].strip()
                    text = ','.join(parts[9:]).strip()

                    # Chỉ lấy tiếng Nhật và bỏ qua nhãn không hợp lệ (###)
                    if language.lower() in ['japanese', 'jpn'] and text != '###':
                        try:
                            # Lấy 4 điểm tọa độ
                            pts = np.array([
                                [int(parts[0]), int(parts[1])],
                                [int(parts[2]), int(parts[3])],
                                [int(parts[4]), int(parts[5])],
                                [int(parts[6]), int(parts[7])]
                            ], dtype=np.int32)

                            # Tính bounding box và cắt ảnh
                            x, y, w, h = cv2.boundingRect(pts)

                            # Bỏ qua nếu khung cắt nằm ngoài ảnh hoặc kích thước bằng 0
                            if x < 0 or y < 0 or w <= 0 or h <= 0: continue
                            crop_img = img[y:y+h, x:x+w]
                            if crop_img.size == 0: continue

                            # Lưu ảnh crop và ghi vào file label mới
                            crop_filename = f'crop_{crop_count:06d}.jpg'
                            cv2.imwrite(os.path.join(rec_img_dir, crop_filename), crop_img)
                            f_out.write(f"images/{crop_filename}\t{text}\n")
                            crop_count += 1
                        except ValueError:
                            pass

print(f"✅ Đã tạo thành công bộ dữ liệu Recognition với {crop_count} ảnh con!")

Bắt đầu xử lý và cắt (crop) ảnh tiếng Nhật...


KeyboardInterrupt: 

In [ ]:
import yaml
import os

config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 1. Cấu hình model và từ điển
config['Global']['pretrained_model'] = './pretrain_models/japan_PP-OCRv3_rec_train/best_accuracy'
config['Global']['character_dict_path'] = 'ppocr/utils/dict/japan_dict.txt'
config['Global']['use_space_char'] = True

# 2. Cấu hình thư mục lưu về Google Drive
os.makedirs('/content/drive/MyDrive/Dict_OCR_Models', exist_ok=True)
config['Global']['save_model_dir'] = '/content/drive/MyDrive/Dict_OCR_Models/rec_japan_v1'

# 3. Trỏ đường dẫn tới Dataset thực tế (đã tạo ở Ô 3)
config['Train']['dataset']['data_dir'] = '/content/PaddleOCR/train_data_rec/'
config['Train']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data_rec/rec_label.txt']
# Nếu lúc train bị lỗi hết RAM (OOM), bạn hãy giảm batch_size xuống (ví dụ 64 hoặc 32)
config['Train']['loader']['batch_size_per_card'] = 128

config['Eval']['dataset']['data_dir'] = '/content/PaddleOCR/train_data_rec/'
config['Eval']['dataset']['label_file_list'] = ['/content/PaddleOCR/train_data_rec/rec_label.txt']
config['Eval']['loader']['batch_size_per_card'] = 128

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f)

print("✅ Đã cập nhật file cấu hình YAML!")

In [30]:
# 1. Xóa sạch bản paddlepaddle cũ (nếu có) để tránh xung đột
!python -m pip uninstall -y paddlepaddle paddlepaddle-gpu

# 2. Cài đặt CỤ THỂ phiên bản paddlepaddle-gpu dành cho CUDA 11.8
# Sử dụng trực tiếp wheel link từ Baidu để đảm bảo chính xác 100%
!python -m pip install paddlepaddle-gpu==2.6.0.post112 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

# 3. Cài đặt các thư viện phụ trợ (nếu chưa cài)
%cd /content/PaddleOCR
!pip install -r requirements.txt
!pip install rapidfuzz pyclipper lmdb

Found existing installation: paddlepaddle-gpu 2.6.2
Uninstalling paddlepaddle-gpu-2.6.2:
  Successfully uninstalled paddlepaddle-gpu-2.6.2
Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 MB 3.1 MB/s eta 0:00:00
/content/PaddleOCR
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment


In [13]:
import yaml
import os

config_path = '/content/PaddleOCR/configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# ---------------------------------------------------------
# 1. TỐI ƯU HÓA HIỆU SUẤT (Tăng tốc độ Train)
# ---------------------------------------------------------
# T4 GPU có 15GB RAM, model Mobile rất nhẹ nên batch size 64 hoặc 128 là cực kỳ tối ưu.
config['Train']['loader']['batch_size_per_card'] = 64
config['Eval']['loader']['batch_size_per_card'] = 64

# ---------------------------------------------------------
# 2. ĐIỀU CHỈNH EPOCH CHỐNG OVERFITTING & COLAB TIMEOUT
# ---------------------------------------------------------
config['Global']['epoch_num'] = 100       # Giảm từ 500 xuống 100 (Fine-tune không cần quá nhiều)
config['Global']['save_epoch_step'] = 5   # Cứ xong 5 epoch sẽ lưu model lên Drive 1 lần
config['Global']['eval_batch_step'] = [0, 500] # Đánh giá mô hình sau mỗi 500 bước

# ---------------------------------------------------------
# 3. TỰ ĐỘNG NỐI TIẾP HUẤN LUYỆN (AUTO-RESUME)
# ---------------------------------------------------------
# Cấu hình này cực kỳ quan trọng trên Colab. Nếu bạn bị văng mạng,
# khi chạy lại code này, nó sẽ tự động tìm model đang train dở trên Drive để chạy tiếp.
save_dir = config['Global'].get('save_model_dir', '/content/drive/MyDrive/Dict_OCR_Models/rec_japan_v1')
checkpoint_path = os.path.join(save_dir, 'latest')

if os.path.exists(f"{checkpoint_path}.pdparams"):
    config['Global']['checkpoints'] = checkpoint_path
    print(f"🔄 Đã tìm thấy Checkpoint! Sẽ Resume huấn luyện từ: {checkpoint_path}")
else:
    config['Global']['checkpoints'] = None

# ---------------------------------------------------------
# 4. GIỮ CÁC CHỐT CHẶN AN TOÀN CHO COLAB
# ---------------------------------------------------------
# Giữ nguyên việc tắt đa luồng để tránh đụng độ bộ nhớ dùng chung (nguyên nhân gây sập C++)
config['Train']['loader']['num_workers'] = 0
config['Eval']['loader']['num_workers'] = 0
config['Train']['loader']['use_shared_memory'] = False
config['Eval']['loader']['use_shared_memory'] = False

# Tắt TRT và MKLDNN (Dù 2 cái này chủ yếu tác động lúc Inference, tắt đi cho chắc cốp)
config['Global']['use_tensorrt'] = False
config['Global']['use_mkldnn'] = False

# Lưu lại file
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f)

print("✅ Đã cập nhật file cấu hình YAML: Tối ưu Batch Size, Cài đặt Epoch hợp lý & Kích hoạt Auto-Resume!")

✅ Đã cập nhật file cấu hình YAML: Tối ưu Batch Size, Cài đặt Epoch hợp lý & Kích hoạt Auto-Resume!


In [36]:
# 1. Gỡ bỏ bản PaddlePaddle CUDA 11 vừa cài
!python -m pip uninstall -y paddlepaddle paddlepaddle-gpu

# 2. Cài đặt bản PaddlePaddle-GPU mới nhất (tương thích CUDA 12.0+ của Colab hiện tại)
!python -m pip install paddlepaddle-gpu==2.6.1.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html

# 3. Kiểm tra lại xem Paddle đã nhận diện thành công GPU chưa
import paddle
paddle.utils.run_check()
print(f"Paddle version: {paddle.__version__}")
print(f"GPU is available: {paddle.is_compiled_with_cuda()}")

Found existing installation: paddlepaddle-gpu 2.6.0.post112
Uninstalling paddlepaddle-gpu-2.6.0.post112:
  Successfully uninstalled paddlepaddle-gpu-2.6.0.post112
Looking in links: https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 MB 1.8 MB/s eta 0:00:00
Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
Paddle version: 2.6.1
GPU is available: True


In [40]:
# 1. DỌN DẸP XUNG ĐỘT OPENCV
# Gỡ cài đặt toàn bộ các bản OpenCV hiện có để tránh đụng độ thư viện C++
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
# Chỉ cài lại bản headless (dành cho môi trường server/Colab, không chứa giao diện UI)
!pip install opencv-python-headless==4.8.0.76

# 2. VÁ LỖI CUDNN 8.9 vs 9.8
# Cài đặt thư viện cuDNN 8.9 cục bộ qua pip để tương thích với Paddle 2.6.1
!pip install nvidia-cudnn-cu12==8.9.2.26

# Ép hệ thống ưu tiên sử dụng thư viện cuDNN 8.9 vừa cài thay vì bản 9.8 mặc định của Colab
import os
import nvidia.cudnn

# Lấy đường dẫn chứa file thư viện cuDNN cục bộ
cudnn_path = os.path.dirname(nvidia.cudnn.__file__)

# Cập nhật biến môi trường LD_LIBRARY_PATH
os.environ['LD_LIBRARY_PATH'] = f"{cudnn_path}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')

print("✅ Đã xử lý xong OpenCV và ép dùng cuDNN tương thích!")
print("Đường dẫn lib hiện tại:", os.environ['LD_LIBRARY_PATH'])

Found existing installation: opencv-python-headless 4.8.0.76
Uninstalling opencv-python-headless-4.8.0.76:
  Successfully uninstalled opencv-python-headless-4.8.0.76
  Using cached opencv_python_headless-4.8.0.76-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
Using cached opencv_python_headless-4.8.0.76-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.8.0.76 which is incompatible.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.8.0.76 which is incompatible.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C
✅ Đã xử lý xong OpenCV và ép dùng cuDNN tương thích!
Đường dẫn lib hiện tại: /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/lib64-nvidia


In [2]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires opencv-python>=3.4.8.29, which is not installed.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.

In [14]:
# 1. TÌM VÀ TIÊU DIỆT HOÀN TOÀN cuDNN 9.8
!rm -f /usr/lib/x86_64-linux-gnu/libcudnn*.so.9*
!rm -f /usr/local/cuda/lib64/libcudnn*.so.9*
!rm -f /usr/lib/x86_64-linux-gnu/libcudnn*.so

# 2. TẠO LẠI ĐƯỜNG DẪN MẶC ĐỊNH ÉP VỀ BẢN 8.9
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn.so.8 /usr/lib/x86_64-linux-gnu/libcudnn.so
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_ops_infer.so.8 /usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so
!ln -s /usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/libcudnn_cnn_infer.so.8 /usr/lib/x86_64-linux-gnu/libcudnn_cnn_infer.so

# Cập nhật lại linker của hệ điều hành
!ldconfig

# 3. CHẠY LẠI HUẤN LUYỆN
%cd /content/PaddleOCR
!python tools/train.py -c configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libum

In [20]:
!python tools/export_model.py \
  -c configs/rec/PP-OCRv3/multi_language/japan_PP-OCRv3_mobile_rec.yml \
  -o Global.pretrained_model=/content/drive/MyDrive/Dict_OCR_Models/rec_japan_v1/best_accuracy \
  Global.save_inference_dir=/content/drive/MyDrive/Dict_OCR_Models/rec_japan_inference \
  Global.export_with_pir=False

Skipping import of the encryption module.
W0505 13:19:55.688663 65883 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.0
W0505 13:19:55.689862 65883 gpu_resources.cc:164] device: 0, cuDNN Version: 8.9.
[2026/05/05 13:19:56] ppocr INFO: load pretrain successful from /content/drive/MyDrive/Dict_OCR_Models/rec_japan_v1/best_accuracy
[2026/05/05 13:19:56] ppocr INFO: Export inference config file to /content/drive/MyDrive/Dict_OCR_Models/rec_japan_inference/inference.yml
Skipping import of the encryption module
I0505 13:19:57.588892 65883 program_interpreter.cc:212] New Executor is Running.
[2026/05/05 13:19:57] ppocr INFO: inference model is saved to /content/drive/MyDrive/Dict_OCR_Models/rec_japan_inference/inference


In [19]:
import paddle
print(paddle.__version__)

2.6.1
